# Unfreezing the SigLIP tower in `laya-vision` (Kaggle GPU T4 ×2 DDP)

[![Encoder](https://img.shields.io/badge/%F0%9F%A4%97%20Encoder-ModernVBERT%2Fmodernvbert-orange)](https://huggingface.co/ModernVBERT/modernvbert)

This notebook runs **initiative 2**, `plans/2-SigLIP-tuning/siglip-tuning.md`. The first trained
`laya-vision` checkpoint reads images genuinely — remove the image and CIFAR-100 falls from 0.745 to
0.060, RVL-CDIP from 0.305 to 0.000 — yet it still loses to SigLIP2 zero-shot at recognition:

| trained task | laya-vision (frozen tower) | SigLIP2 zero-shot | blind |
|---|---|---|---|
| CIFAR-100 | 0.768 | **0.870** | 0.060 |
| RVL-CDIP | 0.342 | **0.422** | 0.000 |
| VQAv2 yes/no | **0.670** | 0.524 | 0.560 |

Throughout that run the SigLIP tower was frozen, so only the 9.4M connector and the 149M text encoder
adapted, on 24k images. **The question is binary: undertrained, or a structural limit of the 64-token
connector?** This notebook answers it by letting the tower train, and changes nothing else — data,
seed, epochs and effective batch are held at initiative 1's values, or the comparison is void.

The answer is decided by the **pre-registered rule in section 9**, which is written before any
number exists and is evaluated by code, not by reading the table afterwards.

**Nothing here is published.** The code lives on the `sb/vision` branch, not on PyPI, so section 2
clones the repo.

---

### ⚠️ Kaggle notebook settings
In the right-hand sidebar under **Notebook options**:
* **Accelerator:** `GPU T4 x2`
* **Internet:** `On`

### Order of work, and what it costs

| section | stage | GPU time |
|---|---|---|
| 3–4 | smoke tests, single GPU then DDP | ~10 min |
| 5 | build and cache the data (no GPU needed — see below) | — |
| 6 | **E5**, learning-rate probe: 2e-6 / 5e-6 / 2e-5 + a frozen control | ~50 min |
| 7 | **E6**, the full run at the winning LR | ~90 min |
| 8 | **E7**, benchmark both arms | ~30 min |

Run the smoke tests. DDP is where the last two training bugs appeared, and both of them would have
cost an hour of a real run.

## 1. Environment & dual-T4 check

In [ ]:
!nvidia-smi
import os, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Right sidebar -> Notebook options -> Accelerator -> GPU T4 x2."
)

# transformers probes for TensorFlow at import; TF's abseil runtime can deadlock model
# construction. Laya is torch-only.
os.environ.update(USE_TF="0", USE_TORCH="1", TOKENIZERS_PARALLELISM="false")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")

## 2. Install

`laya-vision` needs **transformers >= 5.3** (the first release with `ModernVBertModel`) and pillow —
the `laya[vision]` extra. Kaggle images ship transformers 4.x, so this upgrades it.

Set `REPO_URL` to your fork if you are not using the branch below.

In [ ]:
REPO_URL = "https://github.com/sebastianberns/laya.git"   # <- your fork, if different
BRANCH   = "sb/vision"
WORKDIR  = "/kaggle/working/laya"

import os, shutil
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)          # a clean clone each session avoids a stale checkout
!git clone -q -b {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log --oneline -1

!pip install -q -e ".[vision]" "datasets>=3.0.0"

### ⚠️ Restart the session now

`pip install -e .` points at the working tree, but a kernel that already imported `transformers` or
`laya` keeps the **old** modules for its whole life — which silently makes later cells run stale code.

**Run ▸ Restart session**, then continue from the next cell.

In [ ]:
import os
os.environ.update(USE_TF="0", USE_TORCH="1", TOKENIZERS_PARALLELISM="false")
%cd /kaggle/working/laya

import laya, transformers, torch
print("laya         :", laya.__version__, "|", laya.__file__)
print("transformers :", transformers.__version__, "(needs >= 5.3)")
print("torch        :", torch.__version__)
from transformers import ModernVBertConfig      # fails loudly if transformers is too old
print("ModernVBERT support: OK")

## 3. Smoke test — is the tower actually trainable?

The claim this whole initiative rests on is *which parameters moved*, so check that first, on 12
examples per task and 3 optimiser steps. Both arms run: the default (frozen, byte-identical to
initiative 1) and `--vision-lr 5e-6`.

**What to look for**, in the `trainable:` lines:

```
frozen    trainable: 2 optimiser groups | encoder(text+connector) 158.48M @ 2.5e-05 | head 14.97M @ 1.0e-04
                     SigLIP tower frozen (0.00M of 93.52M trainable)
unfrozen  trainable: 3 optimiser groups | encoder(text+connector) 158.48M @ 2.5e-05 | head 14.97M @ 1.0e-04
                     SigLIP tower @ 5.0e-06 (93.52M of 93.52M trainable)
```

Two groups against three, and 0.00M against 93.52M. The encoder figure must be **identical** in both:
the tower moves into its own optimiser group, it does not get added to the encoder's.

Each run also prints a `val:` line per epoch — per-task accuracy on the held-out val split.
Meaningless after 3 steps; it is the plumbing that is being checked.

In [ ]:
SMOKE = ("python research/scripts/train_vision.py --out /kaggle/working/lv_smoke_%s"
         " --smoke --tasks cifar100,vqav2_yesno,typed %s")

for name, extra in (("frozen", ""), ("unfrozen", "--vision-lr 5e-6")):
    print("=" * 30, name)
    cmd = SMOKE % (name, extra)
    print(cmd)
    !{cmd}

## 4. Smoke test — 2×T4 DDP

The part worth watching: NCCL init, and `find_unused_parameters` now that a text-only replay batch
leaves the *whole tower* without gradients rather than just the connector. That flag was merely
prudent before; with `--vision-lr` it is load-bearing, and this cell is what proves it holds.

If it hangs at startup rather than erroring, re-run with `--nproc_per_node=1` to confirm it is the
distributed path.

In [ ]:
cmd = ("torchrun --standalone --nproc_per_node=2 research/scripts/train_vision.py"
       " --out /kaggle/working/lv_ddp --smoke --tasks cifar100,typed --vision-lr 5e-6")
print(cmd)
!{cmd}

## 5. Run configuration and data

Everything both the prepare step and every training run below need. Cache files are keyed by
**task, split, size and seed**, so `TASKS`, `N_PER_TASK` and `SEED` have to be identical across them,
or a run quietly rebuilds the data instead of loading it.

`MICRO_BATCH` and `GRAD_ACCUM` are **not** free choices here: the effective batch is
`MICRO_BATCH × GRAD_ACCUM × 2 GPUs`, initiative 1 trained at 64, and a different one makes this
experiment incomparable to the baseline it is being judged against. If a T4 runs out of memory with
the tower trainable, halve `MICRO_BATCH` and double `GRAD_ACCUM` — that keeps 64.

**Preparing the data costs no GPU time.** `--prepare-only` touches no model weights, so it runs in a
CPU-only Kaggle session: run it there, **Save Version** so its output becomes a dataset, then
**Add Data ▸ Your Datasets** here and point `DATA_CACHE` at
`/kaggle/input/<dataset-name>/vision_data`. About 0.9 GB for the four tasks below.

In [ ]:
# ----- shared by the prepare step and every run below -----
TASKS      = "cifar100,rvl_cdip,vqav2_yesno,typed"   # exactly initiative 1's set: koniq stays out
DATA_CACHE = "/kaggle/working/vision_data"           # or /kaggle/input/<dataset>/vision_data
SEED       = 42
N_PER_TASK = None        # None = the script's per-task defaults (8000, and 5400 for typed)

EPOCHS      = 3
MICRO_BATCH = 8          # x GRAD_ACCUM x 2 GPUs = 64, as initiative 1. Comparability depends on it.
GRAD_ACCUM  = 4
WORKERS     = 4          # DataLoader workers per rank; raise if the GPUs go idle during training

PROBE_N     = 3000       # E5 runs on a subset: the LR probe does not need the full data
PROBE_LRS   = [2e-6, 5e-6, 2e-5]      # the text side runs at 2.5e-5

def data_args(n_per_task=None, cache=None):
    s = f"--tasks {TASKS} --data-cache {cache or DATA_CACHE} --seed {SEED}"
    return s + (f" --n-per-task {n_per_task}" if n_per_task is not None else "")

DATA_ARGS  = data_args(N_PER_TASK)
TRAIN_ARGS = f"--micro-batch {MICRO_BATCH} --grad-accum {GRAD_ACCUM} --workers {WORKERS}"
print(DATA_ARGS, TRAIN_ARGS, sep="\n")

In [ ]:
# Build and cache every split once, in one process: inside torchrun every rank would stream the
# whole thing at once. Both the probe subset (E5) and the full data (E6) are built here.
for args in (data_args(PROBE_N), DATA_ARGS):
    cmd = f"python research/scripts/train_vision.py --out /kaggle/working/prep {args} --prepare-only"
    print(cmd)
    !{cmd}

!du -sh {DATA_CACHE}; ls -la {DATA_CACHE}

## 6. E5 — learning-rate probe

One epoch on 3000 examples per task, at three tower learning rates plus a frozen control on the
**same** subset. ~12 min each.

This exists so that a null result in E6 cannot be blamed on a guessed learning rate. Too high and the
tower forgets what SigLIP2 pretraining gave it; too low and nothing moves. The control is what says
whether any of the three did anything at all.

Decide on **per-task validation accuracy** (the `val:` line, also stored in each checkpoint's
config), not on training loss — loss falls fastest exactly where the tower is overfitting 3000
images.

In [ ]:
PROBE_DIR = "/kaggle/working/probe"
probe_arms = [("frozen", "")] + [(f"lr{lr:g}", f"--vision-lr {lr:g}") for lr in PROBE_LRS]

for name, extra in probe_arms:
    cmd = ("torchrun --standalone --nproc_per_node=2 research/scripts/train_vision.py"
           f" --out {PROBE_DIR}/{name} {data_args(PROBE_N)} {TRAIN_ARGS} --epochs 1 {extra}")
    print("=" * 30, name, "\n", cmd, sep="")
    !{cmd}

In [ ]:
# Every checkpoint records its own learning curve, so the comparison reads the checkpoints
# rather than the scrollback.
import json, os

rows = {}
for name, _ in probe_arms:
    cfg = json.load(open(os.path.join(PROBE_DIR, name, "rl_agent_config.json")))
    rows[name] = (cfg["training"]["val_accuracy"] or [{}])[-1]

tasks = [t for t in sorted({k for r in rows.values() for k in r}) if t != "epoch"]
print("%-10s %s" % ("arm", "".join("%12s" % t for t in tasks)))
for name, r in rows.items():
    print("%-10s %s" % (name, "".join("%12.3f" % r[t] if t in r else "%12s" % "-" for t in tasks)))
print("\nPick the LR that improves the image tasks without costing `typed`; if none beats `frozen`,"
      "\nsay so and run E6 at 5e-6 anyway -- one epoch on a subset is weak evidence of a null.")

## 7. E6 — the full run

3 epochs, full data, the winning learning rate, at initiative 1's effective batch of 64. ~90 min.

Set `VISION_LR` from section 6 before running. Watch the per-epoch `val:` lines: a tower that is
forgetting shows up as `typed` or the image tasks *falling* between epochs, and that is worth
stopping for. `checkpoint_latest/` is written after every epoch, so a Kaggle timeout loses at most
the epoch in progress.

In [ ]:
VISION_LR = 5e-6            # <- set from the E5 table above
OUT_DIR   = "/kaggle/working/laya-vision-unfrozen"

cmd = ("torchrun --standalone --nproc_per_node=2 research/scripts/train_vision.py"
       f" --out {OUT_DIR} {DATA_ARGS} {TRAIN_ARGS} --epochs {EPOCHS} --vision-lr {VISION_LR:g}")
print(cmd)
!{cmd}

## 8. E7 — benchmark

500 held-out test examples per task, seed 0, with the blind baseline on — the images-removed control
is what separates "reads the image better" from "learned the label priors better".

Both arms are scored together when the frozen checkpoint from initiative 1 is available: set
`FROZEN_DIR` to it (add it as a Kaggle dataset, or re-train it with `--vision-lr 0` and otherwise
identical flags). Without it the cell scores the unfrozen arm alone, and section 9 falls back to
initiative 1's recorded numbers — same code, same seed, same `--n`, so the comparison holds, but a
same-session re-score is stronger and worth the disk.

In [ ]:
FROZEN_DIR = None           # <- path to initiative 1's random-init checkpoint, if you have it
REPORT     = "/kaggle/working/vision_benchmark_siglip.json"

arms = f"--model unfrozen={OUT_DIR}"
if FROZEN_DIR:
    arms = f"--model frozen={FROZEN_DIR} " + arms
cmd = f"python research/scripts/bench_vision.py {arms} --n 500 --seed 0 --out {REPORT}"
print(cmd)
!{cmd}

In [ ]:
import json
report = json.load(open(REPORT))
for task, r in report["tasks"].items():
    line = "%-12s (%s)" % (task, r["type"])
    for name in report["models"]:
        if name in r:
            line += "  %s %.3f" % (name, r[name]["accuracy"])
            if "blind" in r[name]:
                line += " (blind %.3f)" % r[name]["blind"]["accuracy"]
    if "siglip2_zero_shot" in r:
        line += "  | siglip2 %.3f" % r["siglip2_zero_shot"]["accuracy"]
    print(line)

## 9. The decision rule, pre-registered

Fixed in `plans/2-SigLIP-tuning/siglip-tuning.md` **before** this ran, and evaluated below by code,
so the outcome cannot be rationalised after seeing the table.

| outcome | rule | what follows |
|---|---|---|
| **Undertrained** | CIFAR-100 ≥ 0.80 **and** RVL-CDIP ≥ 0.40 | continue the line: scale data and epochs, revisit KonIQ |
| **Partial** | at least one improves by ≥ +0.03, but the thresholds are not met | one follow-up run before deciding |
| **Structural limit** | neither improves by ≥ +0.03 | stop; write up the narrower claim |

The plan words the middle row as "exactly one improves by ≥ +0.03". Both improving without either
reaching its threshold is the same situation — real movement, not enough of it — so it is scored
`PARTIAL` too. That reading is fixed here, before the numbers, not after.

**Guardrails**, reported whether or not they bind: `typed` ≥ 0.55 (text decisions must not collapse),
the VQAv2 image contribution ≥ +0.08, and before-temperature ECE no worse than baseline by > 0.05.
Pets and KonIQ are a **forgetting monitor**, not a stop criterion — both are near chance already, and
a drop there is still informative about what tuning the tower costs.

In [ ]:
import json

# initiative 1, E3 random-init arm -- the baseline every rule is written against
BASE     = {"cifar100": 0.768, "rvl_cdip": 0.342, "vqav2_yesno": 0.670, "typed": 0.582,
            "koniq": 0.206, "pets": 0.128}
BASE_ECE = {"cifar100": 0.063, "rvl_cdip": 0.388, "vqav2_yesno": 0.007, "koniq": 0.075,
            "pets": 0.254, "typed": 0.057}
TARGET   = {"cifar100": 0.80, "rvl_cdip": 0.40}
MIN_GAIN = 0.03
ARM      = "unfrozen"

tasks = json.load(open(REPORT))["tasks"]
got = {t: tasks[t][ARM] for t in tasks if ARM in tasks[t]}
base = {t: tasks[t]["frozen"]["accuracy"] if "frozen" in tasks[t] else BASE.get(t) for t in got}

print("%-12s %8s %8s %8s" % ("task", "now", "baseline", "delta"))
for t, m in got.items():
    b = base.get(t)
    print("%-12s %8.3f %8s %8s" % (t, m["accuracy"], "%.3f" % b if b else "-",
                                   "%+.3f" % (m["accuracy"] - b) if b else "-"))

hit      = all(got.get(t, {}).get("accuracy", 0.0) >= v for t, v in TARGET.items())
improved = [t for t in TARGET if t in got and base.get(t) is not None
            and got[t]["accuracy"] - base[t] >= MIN_GAIN]
verdict  = "UNDERTRAINED" if hit else ("PARTIAL" if improved else "STRUCTURAL LIMIT")
print("\nVERDICT: %s  (thresholds met: %s | improved by >= %.2f: %s)"
      % (verdict, hit, MIN_GAIN, improved or "none"))

print("\nguardrails")
if "typed" in got:
    print("  typed >= 0.55                 : %.3f  %s" % (got["typed"]["accuracy"],
                                                          got["typed"]["accuracy"] >= 0.55))
if "vqav2_yesno" in got and "blind" in got["vqav2_yesno"]:
    d = got["vqav2_yesno"]["blind"]["delta"]
    print("  VQAv2 image contribution >=.08: %+.3f  %s" % (d, d >= 0.08))
for t, m in got.items():
    if t in BASE_ECE:
        e = m["before_temp"]["ece"]
        print("  %-12s ECE <= base+0.05: %.3f vs %.3f  %s" % (t, e, BASE_ECE[t], e <= BASE_ECE[t] + 0.05))
print("\nforgetting monitor (zero-shot, reported not scored)")
for t in ("pets", "koniq"):
    if t in got:
        print("  %-6s %.3f (initiative 1: %.3f)" % (t, got[t]["accuracy"], BASE[t]))

## 10. After the run

Whatever the verdict, the deliverables are the same two documents, per `plans/README.md`:

* **`plans/2-SigLIP-tuning/experiments.md`** — one entry per run (E5, E6, E7): hardware, data,
  configuration, the numbers, and what each showed. Copy the `trainable:` and `val:` lines with them;
  they are what makes the run reproducible and what proves the tower trained.
* **`plans/2-SigLIP-tuning/report.md`** — the findings, the scorecard against the decision rule
  above, and the recommendation.

A structural-limit verdict is a **result**: it says laya-vision earns its place where a question
cannot be phrased as an image-text similarity, and not at recognition. Write that, rather than
tuning toward a better-looking number.

Nothing is published from this notebook. `laya-vision` remains unpublished, and the four
pre-publication fixes in the plan land first — `--init-head random` is already the script's default.

Before closing the session: **Save Version** if you want the checkpoints and
`vision_benchmark_siglip.json` to outlive it.